# 以Transformers套件實作問答(Question Answering)功能

In [ ]:
# 載入相關套件
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

In [2]:
# 載入模型
# 新版 transformers（5.x）已移除單純文字抽取式問答的 "question-answering" pipeline，
# 僅保留 document-question-answering、table-question-answering，故改用 AutoModel 手動實作，
# 效果等同於舊版 pipeline("question-answering") 的預設模型 distilbert-base-cased-distilled-squad。
qa_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased-distilled-squad")
qa_model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-cased-distilled-squad")


def nlp(question, context):
    inputs = qa_tokenizer(question, context, return_tensors="pt")
    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_scores = torch.softmax(outputs.start_logits, dim=-1)[0]
    end_scores = torch.softmax(outputs.end_logits, dim=-1)[0]
    start = torch.argmax(start_scores)
    end = torch.argmax(end_scores) + 1

    input_ids = inputs["input_ids"][0]
    answer = qa_tokenizer.decode(input_ids[start:end], skip_special_tokens=True)
    score = (start_scores[start] * end_scores[end - 1]).item()

    return {"answer": answer, "score": score, "start": start.item(), "end": end.item()}

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [3]:
# 訓練資料
context = (
    r"Extractive Question Answering is the task of extracting an answer "
    + "from a text given a question. An example of a question answering "
    + "dataset is the SQuAD dataset, which is entirely based on that task. "
    + "If you would like to fine-tune a model on a SQuAD task, you may "
    + "leverage the examples/question-answering/run_squad.py script."
)

In [4]:
# 測試 2 筆
result = nlp(question="What is extractive question answering?", context=context)
print(
    f"Answer: '{result['answer']}', score: {round(result['score'], 4)}, start: {result['start']}, end: {result['end']}",
)

print()

result = nlp(question="What is a good example of a question answering dataset?", context=context)
print(
    f"Answer: '{result['answer']}', score: {round(result['score'], 4)}, start: {result['start']}, end: {result['end']}",
)

Answer: 'the task of extracting an answer from a text given a question', score: 0.6226, start: 15, end: 28

Answer: 'SQuAD dataset', score: 0.5053, start: 44, end: 50


## 結合Tokenizer

In [5]:
# 結合分詞器(Tokenizer)
tokenizer = AutoTokenizer.from_pretrained("bert-large-uncased-whole-word-masking-finetuned-squad")
model = AutoModelForQuestionAnswering.from_pretrained("bert-large-uncased-whole-word-masking-finetuned-squad")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# 訓練資料
text = r"""
🤗 Transformers (formerly known as pytorch-transformers and pytorch-pretrained-bert) provides general-purpose
architectures (BERT, GPT-2, RoBERTa, XLM, DistilBert, XLNet…) for Natural Language Understanding (NLU) and Natural
Language Generation (NLG) with over 32+ pretrained models in 100+ languages and deep interoperability between
TensorFlow 2.0 and PyTorch.
"""

In [7]:
# 問題
questions = [
    "How many pretrained models are available in 🤗 Transformers?",
    "What does 🤗 Transformers provide?",
    "🤗 Transformers provides interoperability between which frameworks?",
]

In [8]:
# 推測答案
for question in questions:
    inputs = tokenizer(question, text, add_special_tokens=True, return_tensors="pt")
    input_ids = inputs["input_ids"].tolist()[0]

    outputs = model(**inputs)
    answer_start_scores = outputs.start_logits
    answer_end_scores = outputs.end_logits

    # Get the most likely beginning of answer with the argmax of the score
    answer_start = torch.argmax(answer_start_scores)
    # Get the most likely end of answer with the argmax of the score
    answer_end = torch.argmax(answer_end_scores) + 1

    answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(input_ids[answer_start:answer_end]))

    print(f"Question: {question}")
    print(f"Answer: {answer}")

Question: How many pretrained models are available in 🤗 Transformers?
Answer: over 32 +
Question: What does 🤗 Transformers provide?
Answer: general - purpose architectures
Question: 🤗 Transformers provides interoperability between which frameworks?
Answer: tensorflow 2. 0 and pytorch
